# 《PythAPCS123》單元 13-3：執行時期錯誤（Runtime Error, RE）常見排行榜與崩潰防禦

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-3_runtime_errors_ranking_and_defense.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：徹底告別上傳 Online Judge 卻莫名拿到 Runtime Error（RE）的恐懼！深入透視 Python 程式在執行中途突發例外（Exception）而強制中斷的底層機制，地毯式剖析 APCS 考場最常出現的五大 RE 殺手：`IndexError`、`ValueError`、`KeyError`、`ZeroDivisionError` 與 `TypeError`。掌握「走訪前先檢查（Look Before You Leap, LBYL）」的邊界防禦哲學，在存取任何容器與進行危險算術前築起銅牆鐵壁般的條件式護城河，確保程式在各類極端測資下皆能優雅穩健運行。


### 13.3.1 什麼是例外（Exception）？程式執行中途突發崩潰的觸發機制

在前一節中我們學習到，語法錯誤（SyntaxError）發生在直譯器的「編譯剖析期」，只要語法不合規格，整份程式連半個字元都不會執行。而本節要探討的**「執行時期錯誤（Runtime Error，在 OJ 評判系統簡稱為 RE）」**，其性質則截然不同：你的程式碼在文法結構上是完全合法且符合 Python 語法標準的，因此直譯器能夠順利啟動並開始一行行執行。

然而，當程式跑到某一條指令時，突然遭遇了電腦「在邏輯上或物理上無法繼續處理的突發狀況」——例如要求從一個空箱子裡拿東西、要求計算除以零、或是要求把英文字母 `"abc"` 強制轉成十進位整數。此時 Python 直譯器無法憑空猜測該如何往下執行，它唯一的自保手段就是拋出一個**「例外（Exception 物件）」**。如果程式設計師沒有預先寫好攔截防禦措施，直譯器就會當場強制終止程式，並在終端機噴出長串的紅色 Traceback 報錯。在 APCS 考場上，只要有一筆隱藏測資引發未捕捉的例外，該測試點就會直接被判定為 RE，拿到 0 分！

理解「執行時期崩潰」的本質，能幫助我們建立關鍵的除錯意識：程式碼能跑起來不代表永遠不會崩潰；當輸入的測資規模改變、出現空輸入或極端邊界值時，隱藏的例外炸彈就會瞬間引爆。


In [ ]:
# 13.3.1 程式碼演示：對比正常執行與執行中途突發崩潰
print("=== 系統初始化：正常啟動 ===")
numbers = [10, 20, 30]
print(f"目前串列內容: {numbers}")
print(f"成功取出第 0 個元素: {numbers[0]}")
print(f"成功取出第 1 個元素: {numbers[1]}")

print("\n--- 即將執行危險操作：存取不存在的索引 5 ---")
# 為了避免示範時整個執行單元被直接中斷，我們使用 try-except 觀察例外物件的生成
try:
    danger_val = numbers[5]
    print(f"取出數值: {danger_val}")
except IndexError as e:
    print(f"[崩潰現場捕捉] 觸發例外類型: {type(e).__name__}")
    print(f"[錯誤訊息描述] {e}")
    print("說明：Python 在第 11 行遭遇無法處理的越界狀況，拋出 IndexError 例外！")

print("\n=== 系統防禦後維持存活 ===")


### 13.3.1 語法重點回顧與核心觀念提煉

在剛剛的示範中，我們觀察到了執行時期錯誤的三大核心特徵：
1. **前期正常運作**：程式在第 1 行到第 5 行都能如預期印出結果，這證明程式碼本身的語法毫無問題。
2. **即時中斷性**：一旦直譯器遇到無法執行的操作（如 `numbers[5]`），若無防護措施，程式會在出錯的那一行「立刻暴斃」，後續的所有運算與輸出全數被沒收。
3. **物件化例外（Exception Object）**：Python 中的每種執行錯誤都是一個特定的物件類別（例如 `IndexError`, `ValueError`），並附帶詳細的錯誤說明文字。在後續章節中，我們將學習如何運用這些例外名稱進行專門攔截。


In [ ]:
# 13.3.1 學生實作練習：安全存取模擬器
# 任務說明：實作 get_element_safely(lst, idx) 函式
# 若索引 idx 在 lst 的合法範圍內，回傳對應的元素值
# 若索引 idx 超出範圍，回傳字串 "ERROR: Index Out of Bounds" 而不讓程式崩潰

def get_element_safely(lst: list, idx: int):
    # 請在此處撰寫條件判斷進行安全防護
    if 0 <= idx < len(lst):
        return lst[idx]
    return "ERROR: Index Out of Bounds"

# 測試用例
data = ["Apple", "Banana", "Cherry"]
print("合法取值:", get_element_safely(data, 1))
print("越界防護:", get_element_safely(data, 5))
print("負數越界防護:", get_element_safely(data, -1))


In [ ]:
# 13.3.1 單元測試驗證
sample_list = [100, 200, 300]
assert get_element_safely(sample_list, 0) == 100
assert get_element_safely(sample_list, 2) == 300
assert get_element_safely(sample_list, 3) == "ERROR: Index Out of Bounds"
assert get_element_safely(sample_list, -1) == "ERROR: Index Out of Bounds"
assert get_element_safely([], 0) == "ERROR: Index Out of Bounds"
print("13.3.1 單元測試全數通過！")


### 13.3.2 RE 排行榜榜首：IndexError: list index out of range

在 Online Judge 與 APCS 實作題所有送出紀錄中，`IndexError` 長年穩居各類崩潰排行榜的「冠軍寶座」。此錯誤的官方訊息通常為 `IndexError: list index out of range`（串列索引超出範圍），意指你試圖用一個不存在的索引號碼去存取串列元素。

造成 `IndexError` 最主要的常見情境有三大類：
1. **0-based 索引轉換盲點**：串列長度為 $N$ 時，合法的正向索引僅有 $0$ 到 $N-1$。初學者常誤用 `arr[N]` 來試圖讀取「最後一個元素」，直接導致越界崩潰（正確寫法應為 `arr[N-1]` 或 `arr[-1]`）。
2. **走訪邊界差一（Off-by-one）**：在撰寫 `for i in range(len(arr) + 1):` 或是迴圈內比對相鄰元素 `arr[i] == arr[i+1]` 時，最後一輪迭代必定會因為 $i+1 = N$ 而爆炸。
3. **空容器未判斷**：當題目測資可能出現空串列 `[]` 時，直接存取 `arr[0]` 會立刻引爆。此外，在模擬字串處理或佇列操作時，若連續使用 `arr.pop(0)` 直到串列清空後又多做了一次存取，也會瞬間崩潰。

掌握索引防禦的核心心法：在透過索引存取串列前，腦中必須永遠確認「當前串列長度為何？這個索引值是否嚴格落在合法區間內？」


In [ ]:
# 13.3.2 程式碼演示：相鄰元素比對之經典越界陷阱與安全防護
# 需求：檢查串列中是否存在相鄰兩數相同的狀況

danger_data = [1, 3, 5, 5, 8]
print(f"測試陣列: {danger_data}, 長度: {len(danger_data)}")

print("\n--- 錯誤寫法演示（比對到最後一格引發 IndexError）---")
try:
    # 錯誤寫法：range(len(danger_data)) 會走訪到索引 4，而 4+1=5 越界！
    for i in range(len(danger_data)):
        if danger_data[i] == danger_data[i + 1]:
            print(f"找到相鄰重複元素: {danger_data[i]}")
except IndexError as e:
    print(f"[崩潰捕捉] IndexError: {e}，因為最後一輪 i=4 時嘗試存取 danger_data[5]！")

print("\n--- 正確且安全的邊界防禦寫法 ---")
found_dup = False
# 正確寫法：只走訪到倒數第二個元素 len(danger_data) - 1
for i in range(len(danger_data) - 1):
    if danger_data[i] == danger_data[i + 1]:
        print(f"安全找到相鄰重複元素: {danger_data[i]} (位於索引 {i} 與 {i+1})")
        found_dup = True

if not found_dup:
    print("未發現相鄰重複元素")


### 13.3.2 語法重點回顧與核心觀念提煉

防範 `IndexError` 的三大黃金守則：
1. **相鄰比對終點減一**：只要迴圈內會存取 `arr[i + 1]`，則外層的 range 終止條件「絕對」必須縮減為 `range(len(arr) - 1)`；同理，若存取 `arr[i + k]`，終止條件必須為 `range(len(arr) - k)`。
2. **空串列優先過濾**：存取 `arr[0]` 或 `arr[-1]` 前，必須先確認 `if len(arr) > 0:`（或利用 Python 慣用法 `if arr:`）。
3. **切片語法的寬容特性**：需要取出子串列時，切片語法 `arr[a:b]` 即使超出邊界也不會拋出 IndexError，而是會自動截斷至可用範圍，這是撰寫安全取值時常被利用的小技巧。


In [ ]:
# 13.3.2 學生實作練習：安全相鄰差值計算
# 任務說明：實作 calculate_adjacent_diffs(arr) 函式
# 給定一個整數串列 arr，回傳相鄰元素的差值串列 [arr[1]-arr[0], arr[2]-arr[1], ...]
# 邊界要求：若串列長度小於 2，則無法構成相鄰元素，應安全回傳空串列 [] 而非引爆 IndexError！

def calculate_adjacent_diffs(arr: list) -> list:
    # 請先進行長度防禦，再以安全的 range 上界進行迴圈計算
    if len(arr) < 2:
        return []
    diffs = []
    for i in range(len(arr) - 1):
        diffs.append(arr[i + 1] - arr[i])
    return diffs

# 測試用例
print("正常串列差值:", calculate_adjacent_diffs([10, 15, 22, 30]))
print("單一元素邊界:", calculate_adjacent_diffs([100]))
print("空串列邊界:", calculate_adjacent_diffs([]))


In [ ]:
# 13.3.2 單元測試驗證
assert calculate_adjacent_diffs([1, 4, 9, 16]) == [3, 5, 7]
assert calculate_adjacent_diffs([5, 5, 5]) == [0, 0]
assert calculate_adjacent_diffs([42]) == []
assert calculate_adjacent_diffs([]) == []
assert calculate_adjacent_diffs([10, 2]) == [-8]
print("13.3.2 單元測試全數通過！")


### 13.3.3 型態轉換與查找失敗：ValueError

在 APCS 考場的第二大 RE 殺手是 `ValueError`。直譯器噴出此錯誤時，代表「函式接收到的引數型態是正確的，但引數的值（Value）內容不符合該函式的處理規範」。

考場最常見的三大 `ValueError` 引爆場景：
1. **字串轉整數失敗（`invalid literal for int()`）**：使用 `int(s)` 時，若字串內含有非數字字元（如小數點 `"3.14"`、空字串 `""`、或英文字母 `"abc"`），直譯器無法將其解讀為整數。特別是從標準輸入 `input().split()` 讀取測資時，若測資行尾含有多餘字元或格式非預期，極易中招。
2. **多變數解包數量不符（`not enough / too many values to unpack`）**：例如寫下 `a, b = input().split()`，但測資該行實際上只給了 1 個數字，或是給了 3 個數字，解包變數與分割項數不一致時便會引爆。
3. **`list.index(x)` 搜尋不存在的元素**：使用 `arr.index(target)` 查找元素下標時，若 `target` 不在串列中，Python 不會像某些語言回傳 `-1`，而是直接拋出 `ValueError: 'x' is not in list` 讓程式當場暴斃！

要避開 `ValueError`，必須善用先驗檢查（例如在 `index()` 前先用 `if x in arr:` 防守）或驗證輸入字串規格。


In [ ]:
# 13.3.3 程式碼演示：常見 ValueError 場景與防禦策略
print("--- 場景 1: int() 轉型失敗示範 ---")
bad_str = "100a"
try:
    num = int(bad_str)
except ValueError as e:
    print(f"[捕捉 ValueError] 無法將 '{bad_str}' 轉為整數: {e}")

# 防禦對策：先用 isdigit() 檢查是否全由數字構成
if bad_str.isdigit():
    num = int(bad_str)
else:
    print(f"[安全防禦] 字串 '{bad_str}' 包含非數字字元，拒絕轉型！")

print("\n--- 場景 2: list.index() 找不到元素示範 ---")
animals = ["cat", "dog", "rabbit"]
target = "lion"

try:
    pos = animals.index(target)
except ValueError as e:
    print(f"[捕捉 ValueError] 查找失敗: {e}")

# 防禦對策：先以 in 關鍵字做成員檢查
if target in animals:
    pos = animals.index(target)
    print(f"找到 {target} 位於索引 {pos}")
else:
    print(f"[安全防禦] {target} 不在串列中，安全回傳 -1")


### 13.3.3 語法重點回顧與核心觀念提煉

防禦 `ValueError` 的必備工具箱：
1. **整數字串驗證**：在呼叫 `int(s)` 前，若字串可能包含雜訊，可使用 `s.isdigit()` 檢查（注意：負數帶負號 `"-5"` 時 `isdigit()` 會回傳 `False`，若需支援負數，可先檢查 `s.lstrip('-').isdigit()`）。
2. **搜尋成員防守原則**：永遠不要在未確定元素是否存在的情況下直接呼叫 `arr.index(target)`！標準的防守模板是：
   ```python
   idx = arr.index(target) if target in arr else -1
   ```
3. **安全解包處理**：若輸入行包含不定個數的整數，不要寫 `a, b = map(int, input().split())`，而是應該先用串列接收 `tokens = list(map(int, input().split()))`，再檢查 `len(tokens)` 是否符合預期。


In [ ]:
# 13.3.3 學生實作練習：安全整數轉換與搜尋
# 任務說明：實作 find_item_position(items, target) 函式
# 若 target 存在於 items 串列中，回傳其第一次出現的索引
# 若 target 不存在，回傳 -1，絕對不能讓程式噴出 ValueError！

def find_item_position(items: list, target) -> int:
    # 請在此處實作安全查找邏輯
    if target in items:
        return items.index(target)
    return -1

# 測試用例
data_list = ["apple", "banana", "cherry", "durian"]
print("尋找 banana:", find_item_position(data_list, "banana"))
print("尋找 grape:", find_item_position(data_list, "grape"))


In [ ]:
# 13.3.3 單元測試驗證
test_items = [10, 20, 30, 40, 50]
assert find_item_position(test_items, 30) == 2
assert find_item_position(test_items, 10) == 0
assert find_item_position(test_items, 99) == -1
assert find_item_position([], 5) == -1
print("13.3.3 單元測試全數通過！")


### 13.3.4 字典與鍵值遺漏：KeyError 與 .get() 安全查詢防禦

在 Python 中，字典（dict）是解決頻率統計、快速查表與動態對照的核心利器。然而，當你使用中括號語法 `dict[key]` 去存取一個「字典中根本不存在的鍵（Key）」時，Python 會立刻引爆 `KeyError` 異常！

在 APCS 考場上，`KeyError` 最常發生在以下兩大場景：
1. **計數器未初始化**：例如計算字母或單字出現次數時，初學者常直接寫下 `counts[word] += 1`。當第一次遇到某個全新的 `word` 時，字典裡根本還沒有這個鍵，執行 `counts[word]` 試圖讀取舊值時瞬間拋出 `KeyError`。
2. **對照表查詢缺漏**：使用預先定義的字典對照表（例如字元轉密碼、學生學號查成績）時，測資輸入了題目範例中未曾出現的生僻字元或未知代碼，直接用中括號取值立刻導致 RE。

針對 `KeyError`，Python 提供了極其優雅且高效的安全查詢方法：**`.get(key, default_value)`**。當指定的鍵存在時，`.get()` 會回傳對應的 Value；而當鍵不存在時，它不會引爆崩潰，而是平靜地回傳你指定的預設值（若未指定則回傳 `None`）。


In [ ]:
# 13.3.4 程式碼演示：KeyError 觸發與 .get() 安全防禦對比
scores = {"Alice": 95, "Bob": 88, "Charlie": 92}
print(f"現有成績字典: {scores}")

print("\n--- 危險查詢示範 ---")
query_name = "David"
try:
    val = scores[query_name]
    print(f"{query_name} 的成績是: {val}")
except KeyError as e:
    print(f"[捕捉 KeyError] 字典中查無此鍵: {e}")

print("\n--- 防禦方案一：以 in 關鍵字先驗檢查 ---")
if query_name in scores:
    print(f"成績: {scores[query_name]}")
else:
    print(f"[安全防禦] {query_name} 不在字典中，使用預設成績 0 分")

print("\n--- 防禦方案二：使用 dict.get() 優雅取值（考場推薦神技）---")
david_score = scores.get(query_name, 0)
alice_score = scores.get("Alice", 0)
print(f"查詢 David 成績 (不存在，回傳預設值): {david_score}")
print(f"查詢 Alice 成績 (存在，回傳真實值): {alice_score}")


### 13.3.4 語法重點回顧與核心觀念提煉

在競賽與演算法實作中，防禦 `KeyError` 的三大關鍵心法：
1. **中括號取值 vs .get() 取值**：
   - `d[key]`：確定鍵值必定存在時使用，若不存在會引爆 `KeyError`。
   - `d.get(key, default)`：無法保證鍵值是否存在時必用，安全且語法精簡。
2. **計數器初始化的經典寫法**：
   ```python
   # 傳統防禦寫法（需 4 行）：
   if item in counts:
       counts[item] += 1
   else:
       counts[item] = 1

   # .get() 極速單行寫法（考場首選）：
   counts[item] = counts.get(item, 0) + 1
   ```
3. **善用 `defaultdict`（進階可選）**：標準庫 `collections.defaultdict(int)` 能在存取不存在的鍵時自動填入預設值 0，徹底杜絕 KeyError。


In [ ]:
# 13.3.4 學生實作練習：文字頻率統計器
# 任務說明：實作 count_word_frequencies(words) 函式
# 傳入字串串列 words，統計每個單字出現的次數，並回傳統計字典
# 務必使用 dict.get() 實作安全的次數累加，杜絕任何 KeyError 拋出！

def count_word_frequencies(words: list) -> dict:
    freq = {}
    # 請在此處使用 .get() 完成次數累加
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    return freq

# 測試用例
fruits = ["apple", "banana", "apple", "orange", "banana", "apple"]
print("統計結果:", count_word_frequencies(fruits))


In [ ]:
# 13.3.4 單元測試驗證
res = count_word_frequencies(["a", "b", "a", "c", "b", "a"])
assert res == {"a": 3, "b": 2, "c": 1}
assert count_word_frequencies([]) == {}
assert count_word_frequencies(["test"]) == {"test": 1}
print("13.3.4 單元測試全數通過！")


### 13.3.5 算術與型態衝突：ZeroDivisionError 與 TypeError

除了容器索引與字典鍵值問題，在進行數值運算與型態運算子操作時，有兩種不可忽視的例外常導致考場慘劇：**`ZeroDivisionError`** 與 **`TypeError`**。

#### 1. 除以零與模除零：`ZeroDivisionError`
在數學上，任何數除以零都是未定義的；在 Python 中，不論是浮點數除法 `/`、整數除法 `//`、或是取餘數模除 `%`，只要右側的除數為 `0`，直譯器就會無情拋出 `ZeroDivisionError: division by zero` 或 `integer division or modulo by zero`。
在競賽中，這通常發生在：
- 計算平均值 `total / count` 時，當測試資料剛好沒有符合條件的項目導致 `count == 0`。
- 幾何題目中計算兩點斜率 `(y2 - y1) / (x2 - x1)` 時，兩點剛好垂直導致分母為零。

#### 2. 型態操作衝突：`TypeError`
Python 是一門「強型別（Strongly Typed）」語言，它絕對不會自動幫你把字串與數字進行加號拼接。
常見引爆點：
- 試圖把整數與字串用 `+` 串接：`"分數是: " + score`（若 `score` 是整數，會引爆 `TypeError: can only concatenate str to str`）。
- 試圖將整數視為函式呼叫：`x = 10; x()`（引爆 `TypeError: 'int' object is not callable`）。
- 試圖對不支援長度的物件使用 `len(123)`。


In [ ]:
# 13.3.5 程式碼演示：ZeroDivisionError 與 TypeError 陷阱與防禦
print("--- 場景 1: 計算平均值時的除以零陷阱 ---")
scores = [] # 假設某班級沒有任何學生參加考試
print(f"學生成績清單: {scores}")

try:
    avg = sum(scores) / len(scores)
except ZeroDivisionError as e:
    print(f"[捕捉 ZeroDivisionError] 分母不可為零: {e}")

# 防禦對策：先檢查分母是否大於 0
if len(scores) > 0:
    avg = sum(scores) / len(scores)
else:
    avg = 0.0 # 邊界預設處理
print(f"[安全計算] 平均成績為: {avg}")

print("\n--- 場景 2: 型態拼接衝突 TypeError ---")
rank = 1
try:
    msg = "你的排名是第 " + rank + " 名"
except TypeError as e:
    print(f"[捕捉 TypeError] 型態衝突: {e}")

# 防禦對策：使用 f-string 格式化字串，自動安全轉型
safe_msg = f"你的排名是第 {rank} 名"
print(f"[安全輸出] {safe_msg}")


### 13.3.5 語法重點回顧與核心觀念提煉

杜絕算術與型態例外的兩大神盾：
1. **分母必備守門員**：只要程式中出現任何除號（`/`, `//`, `%`），大腦要立刻亮起紅燈，問自己：「這個分母有沒有任何可能是 0？」如果有，必須強制加上 `if denominator != 0:` 分支，並為除數為 0 的情況設定合理的備案。
2. **全面改用 f-string**：在 Python 3 中，請徹底拋棄舊式的加號 `+` 字串串接語法！使用 `f"數值為 {val}"` 不僅語法簡潔美觀，且 f-string 內部會自動呼叫物件的 `__str__()` 方法將各類數值型態安全轉型，從根本上杜絕 `TypeError: can only concatenate str to str` 的發生。


In [ ]:
# 13.3.5 學生實作練習：安全斜率計算機
# 任務說明：實作 calculate_slope(p1, p2) 函式
# 傳入兩點座標元組 p1=(x1, y1), p2=(x2, y2)
# 斜率公式為 (y2 - y1) / (x2 - x1)
# 若兩點垂直（x1 == x2），斜率為無限大，應安全回傳 None 而非噴出 ZeroDivisionError！

def calculate_slope(p1: tuple, p2: tuple):
    x1, y1 = p1
    x2, y2 = p2
    # 請在此處進行分母防禦
    if x1 == x2:
        return None
    return (y2 - y1) / (x2 - x1)

# 測試用例
print("斜線斜率:", calculate_slope((1, 2), (3, 6)))
print("垂直線斜率:", calculate_slope((2, 5), (2, 9)))


In [ ]:
# 13.3.5 單元測試驗證
assert calculate_slope((0, 0), (2, 4)) == 2.0
assert calculate_slope((1, 1), (3, 1)) == 0.0
assert calculate_slope((5, 10), (5, 20)) is None
assert calculate_slope((-1, -1), (1, 1)) == 1.0
print("13.3.5 單元測試全數通過！")


### 13.3.6 邊界防禦第一原則：在存取前以條件式築起護城河（LBYL）

在軟體工程與競賽程式設計中，有一種廣為人知的防禦哲學稱為**「Look Before You Leap（LBYL，三思而後行）」**。它的核心思維非常純粹：在執行任何可能導致災難的危險動作之前，先用嚴密的 `if` 條件判斷做前置檢查；只有當一切條件完全合法時，才允許程式跨出那一步。

在 APCS 考場上，LBYL 是對抗 Runtime Error 最直接、最迅速且最容易排查的武器。相較於事後收拾殘局，事先預防具有以下三大壓倒性優勢：
1. **防止非預期狀態擴散**：若不在源頭攔截，錯誤值可能會流入後續複雜的演算法管線中，最終演變成極難定位的邏輯錯誤（WA）。
2. **保護演算法極端邊界**：許多圖論或動態規劃題目，最常在 $N=0$ 或 $N=1$ 的微型測資上翻車。只要在主函式入口處加上 2 行邊界條件特殊處理（Edge Case Guard），就能穩穩守住關鍵分數。
3. **保持程式流程清晰單純**：利用守衛條件（Guard Clauses），不合法的輸入在最前線就直接被過濾或回傳，主幹程式碼便可以放心地在理想的假設下專注執行核心演算法。

隨時牢記邊界防禦的口訣：「查索引先看長度、做除法先看分母、查字典先用 get、做字串先看空值」。


In [ ]:
# 13.3.6 程式碼演示：綜合邊界防禦範例（安全陣列元素除法器）
# 需求：從陣列中取出 index_a 與 index_b 的元素相除，回傳結果
# 危險點：索引越界（IndexError）、除數為零（ZeroDivisionError）

def safe_array_divide(arr: list, idx_a: int, idx_b: int):
    print(f"\n[執行請求] 嘗試計算 arr[{idx_a}] / arr[{idx_b}]，陣列長度={len(arr)}")
    
    # 邊界防禦 1: 檢查索引 idx_a 是否合法
    if not (0 <= idx_a < len(arr)):
        print(f"❌ 防禦攔截：idx_a ({idx_a}) 超出範圍！")
        return None
        
    # 邊界防禦 2: 檢查索引 idx_b 是否合法
    if not (0 <= idx_b < len(arr)):
        print(f"❌ 防禦攔截：idx_b ({idx_b}) 超出範圍！")
        return None
        
    # 邊界防禦 3: 檢查分母是否為 0
    if arr[idx_b] == 0:
        print(f"❌ 防禦攔截：分母 arr[{idx_b}] 數值為 0，不可除以零！")
        return None
        
    # 所有條件安全，放行執行
    result = arr[idx_a] / arr[idx_b]
    print(f"✅ 安全計算成功：{arr[idx_a]} / {arr[idx_b]} = {result}")
    return result

test_list = [100, 50, 0, 25]
safe_array_divide(test_list, 0, 1) # 正常計算 100 / 50 = 2.0
safe_array_divide(test_list, 0, 2) # 分母為 0 攔截
safe_array_divide(test_list, 0, 9) # 索引越界攔截


### 13.3.6 語法重點回顧與核心觀念提煉

邊界防禦原則（LBYL）的精髓在於：**主動出擊，防範未然**。
在面對 APCS 各類題目時，養成在動筆寫解題邏輯前的「一分鐘防禦掃描習慣」：
1. **輸入長度為 0 或 1 時會怎樣？**（例如求次大值，若只有 1 個元素能否處理？）
2. **尋找的值不在集合中會怎樣？**
3. **計算比率時分母有沒有可能全為 0？**
只要在程式開頭處寫下幾行優雅的衛語句（Guard Clauses），就能直接把 80% 的潛在 RE 扼殺在搖籃中，讓你的程式碼具備堅不可摧的抗擊穩定度！


In [ ]:
# 13.3.6 學生實作練習：安全陣列中位數計算機
# 任務說明：實作 calculate_median(arr) 函式
# 給定一個整數串列 arr，回傳其排序後的中位數（若長度為奇數取正中央，若為偶數取中央兩數之平均 float）
# 邊界防禦要求：
# 1. 若傳入空串列 []，應安全回傳 None 而非崩潰引爆 IndexError！
# 2. 必須在修改排序時防範副作用，不能破壞傳入的原串列！

def calculate_median(arr: list):
    # 請在此處實作衛語句防禦與中位數計算
    if not arr:
        return None
    sorted_arr = sorted(arr)
    n = len(sorted_arr)
    mid = n // 2
    if n % 2 == 1:
        return float(sorted_arr[mid])
    else:
        return (sorted_arr[mid - 1] + sorted_arr[mid]) / 2.0

# 測試用例
print("奇數長度中位數:", calculate_median([3, 1, 2]))
print("偶數長度中位數:", calculate_median([1, 4, 2, 3]))
print("空串列邊界安全測試:", calculate_median([]))


In [ ]:
# 13.3.6 單元測試驗證
assert calculate_median([]) is None
assert calculate_median([5]) == 5.0
assert calculate_median([1, 3, 2]) == 2.0
assert calculate_median([1, 2, 3, 4]) == 2.5
origin = [10, 5, 8]
calculate_median(origin)
assert origin == [10, 5, 8], "不可竄改外部原串列！"
print("13.3.6 單元測試全數通過！")


## 13.3 總結與考場除錯防禦全景對照表

在本單元中，我們系統性地拆解了直譯器執行時期崩潰（Runtime Error, RE）的發生機制與 APCS 考場最常見的五大例外殺手。牢記以下各類例外的對應防禦手段，是你攀登 APCS 實作滿分的核心防彈衣：

| 例外名稱（Exception） | 觸發根本成因 | 典型災難場景 | 最佳防禦黃金法則 |
| :--- | :--- | :--- | :--- |
| **`IndexError`** | 存取超出範圍之索引 | 迴圈查相鄰 `arr[i+1]`、空串列取 `arr[0]` | 迴圈縮減上界 `len-1`，取值前 `if 0 <= i < len:` |
| **`ValueError`** | 傳入數值內容不合規範 | `int("abc")` 轉型失敗、`arr.index(x)` 查無元素 | 轉型前先驗證、搜尋前先 `if x in arr:` |
| **`KeyError`** | 字典中查無指定之鍵 | 頻率統計未初始化、對照表缺漏 | 存取全面改用 `dict.get(key, default)` |
| **`ZeroDivisionError`** | 除數或模除運算元為零 | 計算平均時計數為 0、計算斜率兩點垂直 | 除法運算前必設守衛：`if denominator != 0:` |
| **`TypeError`** | 型態不支援該運算子 | 字串以 `+` 串接整數、物件非函式被呼叫 | 字串拼接全面升級為 `f"{var}"` 格式化字串 |

### 🚀 下一步學習指引
雖然運用「三思而後行（LBYL）」的 `if` 條件式能擋下絕大多數已知風險，但在真實的競賽環境中，有些狀況難以透過簡單的 `if` 預判——最著名的例子就是 APCS 題目常見的**「未知行數輸入（讀取至檔案結尾 EOF）」**。
在下一單元 **13-4《例外捕捉語法：try ... except 架構與未知長度輸入處理》** 中，我們將正式學習 Python 原生的主動防禦框架 `try ... except`，並掌握考場必備的 `EOFError` 終端讀檔神技！
